# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 35 • Pretrained Transformer Models and Hugging Face Workflows

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 170–210 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson introduces practical workflows for pretrained Transformer models
with the Hugging Face ecosystem. It covers model discovery, model cards,
tokenizers, Auto Classes, pipelines, manual inference, batching, feature
extraction, pooling, checkpoint management, reproducibility, model selection,
and fine-tuning structure.

The notebook has two execution layers:

1. **Offline core:** runs completely without internet or the `transformers`
   package.
2. **Optional Hugging Face demonstrations:** run only when the package is
   installed and the learner enables online or cached-model execution.

This design keeps the lesson CPU-safe and reproducible while still teaching
the real Hugging Face APIs.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the roles of the Hugging Face Hub and Transformers library;
- inspect model cards before selecting a checkpoint;
- distinguish model IDs, revisions, configurations, tokenizers, and weights;
- use pipeline-style and manual inference workflows;
- explain AutoTokenizer and AutoModel classes;
- tokenize single examples and batches;
- interpret `input_ids`, `attention_mask`, and model logits;
- convert logits to probabilities and labels;
- extract contextual features and pool token representations;
- compare frozen-feature and fine-tuning workflows;
- save and reload local checkpoints;
- design CPU-safe experiments;
- identify licensing, bias, language, domain, and data limitations;
- evaluate Arabic and multilingual checkpoint suitability.

## Table of Contents

1. The Hugging Face Ecosystem
2. Hub Repositories and Model IDs
3. Model Cards
4. Model Selection Checklist
5. Installation and Environment Checks
6. Offline and Online Execution Modes
7. Pipelines
8. Auto Classes
9. Tokenizer Outputs
10. Padding and Truncation
11. Batch Inference
12. Model Outputs and Logits
13. Probability Conversion
14. Feature Extraction
15. Token Pooling
16. Frozen Features Versus Fine-Tuning
17. Local CPU-Safe Demonstration Dataset
18. Offline Vocabulary and Tokenizer
19. Dynamic Padding
20. Local Transformer Classifier
21. Training Utilities
22. Training the Offline Classifier
23. Learning Curves
24. Evaluation
25. Confusion Matrix
26. Error Analysis
27. Contextual Feature Extraction
28. Sentence Similarity
29. Saving and Reloading a Checkpoint
30. Optional Pipeline Demonstration
31. Optional Manual Auto-Class Inference
32. Optional Feature-Extraction Demonstration
33. Fine-Tuning Workflow Structure
34. Trainer Workflow Structure
35. Data Collators
36. Checkpoint Revisions and Caching
37. CPU, GPU, and Memory Decisions
38. Responsible Model Selection
39. Arabic and Multilingual Considerations
40. Reproducibility and Reporting
41. Knowledge Check
42. Exercises
43. Summary and Next Lesson

# 1. The Hugging Face Ecosystem

The main components used in this lesson are:

- **Hugging Face Hub:** hosts model, dataset, and application repositories;
- **Transformers:** provides model architectures, tokenizers, pipelines, and
  training utilities;
- **huggingface_hub:** manages Hub downloads, uploads, revisions, and cards;
- **Datasets:** provides dataset loading and transformation workflows;
- **Evaluate:** provides metric-loading workflows.

These components are related but independently installable.

In [ ]:
import copy
import importlib.util
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

ecosystem = pd.DataFrame(
    [
        ("Hub", "repositories and artifacts"),
        ("Transformers", "models, tokenizers, pipelines, training"),
        ("huggingface_hub", "Hub access and repository metadata"),
        ("Datasets", "dataset loading and processing"),
        ("Evaluate", "metric workflows"),
    ],
    columns=["Component", "Primary role"],
)

ecosystem

# 2. Hub Repositories and Model IDs

A model ID commonly has the form:

```text
organization-or-user/repository-name
```

A repository may contain:

- configuration files;
- tokenizer files;
- model weights;
- generation settings;
- model card;
- training metadata;
- license information.

In [ ]:
repository_artifacts = pd.DataFrame(
    [
        ("config.json", "architecture and hyperparameters"),
        ("tokenizer files", "text-to-token conversion"),
        ("model weights", "learned parameters"),
        ("README/model card", "intended use and limitations"),
        ("generation config", "generation defaults"),
    ],
    columns=["Artifact", "Purpose"],
)

repository_artifacts

# 3. Model Cards

A model card should be reviewed before using a checkpoint.

Important sections include:

- model description;
- intended uses;
- training data;
- evaluation;
- limitations;
- bias and risk;
- languages;
- license;
- citation.

In [ ]:
model_card_checklist = pd.DataFrame(
    [
        ("Task", "Does the checkpoint match the intended task?"),
        ("Language", "Are the required languages supported?"),
        ("Domain", "Does training resemble the target domain?"),
        ("License", "Is the intended use permitted?"),
        ("Metrics", "Are evaluations relevant and reproducible?"),
        ("Limitations", "What failures are documented?"),
        ("Revision", "Which exact version will be used?"),
    ],
    columns=["Check", "Question"],
)

model_card_checklist

# 4. Model Selection Checklist

Model popularity alone is not sufficient.

Selection should consider:

- task compatibility;
- language coverage;
- model size;
- context length;
- inference latency;
- memory requirements;
- license;
- documented risks;
- reproducibility.

In [ ]:
candidate_models = pd.DataFrame(
    [
        (
            "Small encoder",
            "classification",
            "low",
            "fast CPU inference",
        ),
        (
            "Multilingual encoder",
            "cross-lingual tasks",
            "medium",
            "broader language coverage",
        ),
        (
            "Large decoder",
            "generation",
            "high",
            "greater memory demand",
        ),
    ],
    columns=[
        "Candidate",
        "Typical task",
        "Relative cost",
        "Selection reason",
    ],
)

candidate_models

# 5. Installation and Environment Checks

The offline core requires:

- Python;
- PyTorch;
- NumPy;
- pandas;
- scikit-learn;
- matplotlib.

The optional demonstrations require `transformers`.

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec(
        "transformers"
    )
    is not None
)

HUB_AVAILABLE = (
    importlib.util.find_spec(
        "huggingface_hub"
    )
    is not None
)

environment_status = pd.Series(
    {
        "Python": platform.python_version(),
        "PyTorch": torch.__version__,
        "Device": "CPU",
        "Transformers installed": TRANSFORMERS_AVAILABLE,
        "huggingface_hub installed": HUB_AVAILABLE,
    }
)

environment_status

In [ ]:
print(
    "CPU installation commands:"
)
print(
    "python -m pip install transformers huggingface_hub"
)
print(
    "PyTorch CPU wheels:"
)
print(
    "python -m pip install torch "
    "--index-url https://download.pytorch.org/whl/cpu"
)

Restart the notebook kernel after installing packages.

# 6. Offline and Online Execution Modes

The optional cells are disabled by default.

Enable them only when:

- `transformers` is installed;
- internet access is available, or the checkpoint is already cached;
- the checkpoint license and model card have been reviewed.

In [ ]:
RUN_HUGGING_FACE_DEMOS = False

DEMO_MODEL_ID = (
    "distilbert/"
    "distilbert-base-uncased-finetuned-sst-2-english"
)

USE_LOCAL_FILES_ONLY = True

optional_status = pd.Series(
    {
        "Run optional demos": RUN_HUGGING_FACE_DEMOS,
        "Model ID": DEMO_MODEL_ID,
        "Local files only": USE_LOCAL_FILES_ONLY,
    }
)

optional_status

Set `USE_LOCAL_FILES_ONLY = False` only when downloads are intentionally
permitted.

# 7. Pipelines

The pipeline API combines preprocessing, model inference, and output
postprocessing.

Conceptual usage:

```python
from transformers import pipeline
classifier = pipeline("text-classification", model=MODEL_ID, device=-1)
classifier("The experiment produced excellent results.")
```

`device=-1` requests CPU execution in commonly used pipeline workflows.

In [ ]:
pipeline_workflow = pd.DataFrame(
    [
        (1, "Select task"),
        (2, "Load tokenizer and model"),
        (3, "Preprocess input"),
        (4, "Run inference"),
        (5, "Postprocess output"),
    ],
    columns=["Stage", "Pipeline action"],
)

pipeline_workflow

# 8. Auto Classes

Auto Classes choose an implementation from checkpoint configuration.

Common classes:

- `AutoTokenizer`;
- `AutoConfig`;
- `AutoModel`;
- `AutoModelForSequenceClassification`;
- `AutoModelForTokenClassification`;
- `AutoModelForMaskedLM`;
- `AutoModelForCausalLM`;
- `AutoModelForSeq2SeqLM`.

In [ ]:
auto_class_table = pd.DataFrame(
    [
        ("AutoModel", "base hidden representations"),
        (
            "AutoModelForSequenceClassification",
            "sequence labels",
        ),
        (
            "AutoModelForTokenClassification",
            "token labels",
        ),
        (
            "AutoModelForMaskedLM",
            "masked-token prediction",
        ),
        (
            "AutoModelForCausalLM",
            "next-token generation",
        ),
        (
            "AutoModelForSeq2SeqLM",
            "conditional generation",
        ),
    ],
    columns=["Class", "Output head"],
)

auto_class_table

# 9. Tokenizer Outputs

Tokenizers commonly return:

- `input_ids`;
- `attention_mask`;
- optionally `token_type_ids`;
- offsets or special-token masks when requested.

In [ ]:
tokenizer_output_shapes = pd.DataFrame(
    [
        ("input_ids", "(batch, sequence_length)"),
        ("attention_mask", "(batch, sequence_length)"),
        ("token_type_ids", "(batch, sequence_length), model-dependent"),
    ],
    columns=["Field", "Typical shape"],
)

tokenizer_output_shapes

# 10. Padding and Truncation

Batch tokenization commonly controls:

- padding policy;
- truncation;
- maximum length;
- tensor framework;
- special tokens.

In [ ]:
tokenization_options = pd.DataFrame(
    [
        ("padding=True", "pad to longest sequence in the batch"),
        ("padding='max_length'", "pad to a fixed maximum"),
        ("truncation=True", "truncate over-length sequences"),
        ("return_tensors='pt'", "return PyTorch tensors"),
    ],
    columns=["Option", "Effect"],
)

tokenization_options

# 11. Batch Inference

Batching improves throughput by processing multiple inputs together.

A batch must use consistent tensor dimensions, which is why padding is
required.

# 12. Model Outputs and Logits

Sequence-classification models commonly return an object containing `logits`.

Logits are unnormalized scores.

```text
logits shape = (batch_size, number_of_labels)
```

In [ ]:
example_logits = torch.tensor(
    [
        [1.8, -0.4],
        [-0.2, 1.3],
    ],
    dtype=torch.float32,
)

example_logits

# 13. Probability Conversion

Softmax converts classification logits into probabilities.

In [ ]:
example_probabilities = torch.softmax(
    example_logits,
    dim=1,
)

pd.DataFrame(
    example_probabilities.numpy(),
    columns=["class_0", "class_1"],
)

Label meanings should be read from model configuration, such as `id2label`.

# 14. Feature Extraction

Base Transformer models produce contextual token representations.

Typical output:

```text
last_hidden_state: (batch, sequence_length, hidden_dimension)
```

# 15. Token Pooling

To produce one vector per sentence, common methods include:

- CLS pooling;
- masked mean pooling;
- maximum pooling;
- learned pooling.

In [ ]:
pooling_methods = pd.DataFrame(
    [
        ("CLS", "use a designated first-token state"),
        ("Mean", "average valid token states"),
        ("Max", "take maximum per feature"),
        ("Learned", "train an attention or projection layer"),
    ],
    columns=["Method", "Mechanism"],
)

pooling_methods

# 16. Frozen Features Versus Fine-Tuning

**Frozen feature extraction**

- keeps pretrained weights fixed;
- trains only a task-specific classifier;
- uses less memory and time;
- may under-adapt to the task.

**Fine-tuning**

- updates some or all pretrained weights;
- often improves task adaptation;
- requires more memory and careful learning rates;
- can overfit or forget pretrained knowledge.

In [ ]:
transfer_comparison = pd.DataFrame(
    [
        (
            "Frozen",
            "classifier only",
            "lower",
            "limited adaptation",
        ),
        (
            "Fine-tuned",
            "encoder and classifier",
            "higher",
            "overfitting or forgetting",
        ),
    ],
    columns=[
        "Strategy",
        "Updated parameters",
        "Relative cost",
        "Main risk",
    ],
)

transfer_comparison

# 17. Local CPU-Safe Demonstration Dataset

The offline experiment uses four balanced classes:

- health;
- finance;
- technology;
- travel.

In [ ]:
records = [
    ("doctor treats patient in hospital", "health"),
    ("nurse provides medicine to patient", "health"),
    ("patient visits clinic for diagnosis", "health"),
    ("hospital schedules medical treatment", "health"),
    ("exercise supports long term health", "health"),
    ("nutrition improves patient recovery", "health"),
    ("doctor reviews the medical report", "health"),
    ("clinic provides emergency service", "health"),
    ("nurse helps the patient today", "health"),
    ("medicine reduces the health problem", "health"),
    ("hospital needs experienced doctors", "health"),
    ("patient requests treatment information", "health"),
    ("medical team monitors recovery", "health"),
    ("clinic updates treatment plan", "health"),
    ("doctor confirms diagnosis", "health"),
    ("patient receives medicine", "health"),

    ("bank approves customer loan", "finance"),
    ("invoice contains payment charge", "finance"),
    ("customer requests card refund", "finance"),
    ("billing account has a problem", "finance"),
    ("loan interest increased today", "finance"),
    ("bank transfers money safely", "finance"),
    ("payment failed on the card", "finance"),
    ("refund request remains pending", "finance"),
    ("invoice price is incorrect", "finance"),
    ("customer updates bank account", "finance"),
    ("billing service changed charge", "finance"),
    ("loan payment needs approval", "finance"),
    ("bank reviews financial request", "finance"),
    ("customer receives refund", "finance"),
    ("payment system confirms transaction", "finance"),
    ("card account shows charge", "finance"),

    ("software update caused an error", "technology"),
    ("application cannot reach server", "technology"),
    ("network upload failed today", "technology"),
    ("computer needs a system update", "technology"),
    ("device cannot install software", "technology"),
    ("server lost important data", "technology"),
    ("application displays network error", "technology"),
    ("computer connects to server", "technology"),
    ("upload request failed again", "technology"),
    ("system update needs technical help", "technology"),
    ("device reports software problem", "technology"),
    ("network service is unavailable", "technology"),
    ("server restarts after update", "technology"),
    ("application recovers after installation", "technology"),
    ("computer stores data", "technology"),
    ("network error interrupts upload", "technology"),

    ("flight arrives at airport", "travel"),
    ("tourist books hotel reservation", "travel"),
    ("airport lost passenger luggage", "travel"),
    ("travel ticket changed today", "travel"),
    ("flight delay affects journey", "travel"),
    ("hotel reservation needs update", "travel"),
    ("tourist visits city museum", "travel"),
    ("beach trip starts tomorrow", "travel"),
    ("airport changes flight gate", "travel"),
    ("passenger requests travel information", "travel"),
    ("journey includes hotel stay", "travel"),
    ("ticket service reports delay", "travel"),
    ("tourist reaches airport", "travel"),
    ("passenger collects luggage", "travel"),
    ("hotel confirms reservation", "travel"),
    ("flight continues after delay", "travel"),
]

dataset = pd.DataFrame(
    records,
    columns=["text", "label"],
)

dataset["label"].value_counts()

# 18. Offline Vocabulary and Tokenizer

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


(
    X_train_full,
    X_test,
    y_train_full,
    y_test,
) = train_test_split(
    dataset["text"],
    dataset["label"],
    test_size=0.25,
    random_state=42,
    stratify=dataset["label"],
)

(
    X_train,
    X_validation,
    y_train,
    y_validation,
) = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

counts = Counter(
    token
    for text in X_train
    for token in tokenize(text)
)

vocabulary = [
    "<PAD>",
    "<UNK>",
    "<CLS>",
] + sorted(counts)

token_to_index = {
    token: index
    for index, token
    in enumerate(vocabulary)
}

PAD_ID = token_to_index["<PAD>"]
UNK_ID = token_to_index["<UNK>"]
CLS_ID = token_to_index["<CLS>"]

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

print("Vocabulary size:", len(vocabulary))
print("Classes:", list(label_encoder.classes_))

This tokenizer is intentionally simple. It provides an offline comparison point
for the pretrained tokenizer workflows introduced earlier.

# 19. Dynamic Padding

In [ ]:
class TextDataset(Dataset):
    def __init__(
        self,
        texts,
        labels,
    ):
        self.texts = list(texts)
        self.labels = label_encoder.transform(
            list(labels)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        token_ids = [
            CLS_ID
        ] + [
            token_to_index.get(
                token,
                UNK_ID,
            )
            for token in tokenize(
                self.texts[index]
            )
        ]

        return {
            "input_ids": torch.tensor(
                token_ids,
                dtype=torch.long,
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
            "text": self.texts[index],
        }


def collate_batch(batch):
    maximum_length = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            maximum_length,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    labels = []

    for row, item in enumerate(batch):
        sequence = item["input_ids"]

        input_ids[
            row,
            :len(sequence),
        ] = sequence

        labels.append(
            item["label"]
        )

    return {
        "input_ids": input_ids,
        "attention_mask": (
            input_ids != PAD_ID
        ).long(),
        "padding_mask": (
            input_ids == PAD_ID
        ),
        "labels": torch.stack(labels),
        "texts": [
            item["text"]
            for item in batch
        ],
    }

In [ ]:
train_dataset = TextDataset(
    X_train,
    y_train,
)
validation_dataset = TextDataset(
    X_validation,
    y_validation,
)
test_dataset = TextDataset(
    X_test,
    y_test,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
)

sample_batch = next(iter(train_loader))

print(
    "input_ids:",
    sample_batch["input_ids"].shape,
)
print(
    "attention_mask:",
    sample_batch["attention_mask"].shape,
)

# 20. Local Transformer Classifier

This compact model mirrors the shape conventions used by pretrained encoder
classifiers.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        maximum_length: int = 128,
    ):
        super().__init__()

        encoding = torch.zeros(
            maximum_length,
            model_dimension,
        )

        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(10000.0)
                / model_dimension
            )
        )

        encoding[:, 0::2] = torch.sin(
            positions * rates
        )
        encoding[:, 1::2] = torch.cos(
            positions * rates
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        return (
            embeddings
            + self.encoding[
                :,
                :embeddings.size(1),
                :,
            ]
        )


class LocalTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        class_count: int,
        model_dimension: int = 32,
        head_count: int = 4,
        layer_count: int = 2,
        feed_forward_dimension: int = 64,
        dropout: float = 0.15,
    ):
        super().__init__()

        self.model_dimension = model_dimension

        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position = PositionalEncoding(
            model_dimension
        )

        layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=(
                feed_forward_dimension
            ),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=layer_count,
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            model_dimension,
            class_count,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        embeddings = (
            self.embedding(input_ids)
            * math.sqrt(
                self.model_dimension
            )
        )

        encoded = self.encoder(
            self.position(embeddings),
            src_key_padding_mask=(
                padding_mask
            ),
        )

        representation = encoded[
            :,
            0,
            :,
        ]

        logits = self.classifier(
            self.dropout(
                representation
            )
        )

        return {
            "logits": logits,
            "last_hidden_state": encoded,
            "pooler_output": representation,
        }


DEVICE = torch.device("cpu")

torch.manual_seed(42)

local_model = LocalTransformerClassifier(
    vocabulary_size=len(vocabulary),
    class_count=len(label_encoder.classes_),
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter
        in local_model.parameters()
    ),
)

The output dictionary uses names similar to common pretrained model outputs.

# 21. Training Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


loss_function = nn.CrossEntropyLoss()


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
):
    model.eval()

    losses = []
    labels_all = []
    predictions_all = []
    probabilities_all = []
    texts_all = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            output = model(
                input_ids,
                padding_mask,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            probabilities = torch.softmax(
                output["logits"],
                dim=1,
            )

            predictions = probabilities.argmax(
                dim=1
            )

            losses.append(
                float(loss.item())
            )
            labels_all.extend(
                labels.cpu().tolist()
            )
            predictions_all.extend(
                predictions.cpu().tolist()
            )
            probabilities_all.extend(
                probabilities.cpu().tolist()
            )
            texts_all.extend(
                batch["texts"]
            )

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(
            labels_all,
            predictions_all,
        ),
        "macro_f1": f1_score(
            labels_all,
            predictions_all,
            average="macro",
        ),
        "labels": np.asarray(labels_all),
        "predictions": np.asarray(
            predictions_all
        ),
        "probabilities": np.asarray(
            probabilities_all
        ),
        "texts": texts_all,
    }

# 22. Training the Offline Classifier

In [ ]:
def train_model(
    model: nn.Module,
    epochs: int = 50,
    learning_rate: float = 0.004,
    patience: int = 9,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_loss = float(
        "inf"
    )
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            optimizer.zero_grad()

            output = model(
                input_ids,
                padding_mask,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            loss.backward()

            gradient_norm = clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            training_losses.append(
                float(loss.item())
            )
            gradient_norms.append(
                float(gradient_norm)
            )

        validation_metrics = evaluate_model(
            model,
            validation_loader,
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    np.mean(
                        training_losses
                    )
                ),
                "validation_loss": (
                    validation_metrics[
                        "loss"
                    ]
                ),
                "validation_accuracy": (
                    validation_metrics[
                        "accuracy"
                    ]
                ),
                "validation_macro_f1": (
                    validation_metrics[
                        "macro_f1"
                    ]
                ),
                "gradient_norm": float(
                    np.mean(
                        gradient_norms
                    )
                ),
            }
        )

        if (
            validation_metrics["loss"]
            < best_validation_loss
            - 1e-5
        ):
            best_validation_loss = (
                validation_metrics[
                    "loss"
                ]
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            without_improvement = 0
        else:
            without_improvement += 1

        if (
            without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return model, pd.DataFrame(
        history
    )


set_seed(42)

trained_local_model, training_history = (
    train_model(local_model)
)

print(
    "Epochs completed:",
    len(training_history),
)
print(
    "Best validation macro F1:",
    round(
        training_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 23. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history[
        "training_loss"
    ],
    label="Training loss",
)
plt.plot(
    training_history["epoch"],
    training_history[
        "validation_loss"
    ],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Offline Transformer Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history[
        "validation_macro_f1"
    ],
)
plt.xlabel("Epoch")
plt.ylabel("Validation macro F1")
plt.title("Validation Macro F1")
plt.tight_layout()
plt.show()

# 24. Evaluation

In [ ]:
test_metrics = evaluate_model(
    trained_local_model,
    test_loader,
)

print(
    "Test accuracy:",
    round(
        test_metrics["accuracy"],
        3,
    ),
)
print(
    "Test macro F1:",
    round(
        test_metrics["macro_f1"],
        3,
    ),
)

actual_labels = (
    label_encoder.inverse_transform(
        test_metrics["labels"]
    )
)

predicted_labels = (
    label_encoder.inverse_transform(
        test_metrics[
            "predictions"
        ]
    )
)

print(
    classification_report(
        actual_labels,
        predicted_labels,
        zero_division=0,
    )
)

# 25. Confusion Matrix

In [ ]:
class_names = list(
    label_encoder.classes_
)

matrix = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=class_names,
)

pd.DataFrame(
    matrix,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 26. Error Analysis

In [ ]:
error_frame = pd.DataFrame(
    {
        "text": test_metrics["texts"],
        "actual": actual_labels,
        "predicted": predicted_labels,
        "confidence": test_metrics[
            "probabilities"
        ].max(axis=1),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

Review errors for:

- unknown words;
- ambiguous short inputs;
- mixed-domain vocabulary;
- overconfidence;
- class imbalance;
- domain shift.

# 27. Contextual Feature Extraction

In [ ]:
def encode_for_local_model(
    texts: list[str],
):
    examples = [
        {
            "input_ids": torch.tensor(
                [
                    CLS_ID
                ] + [
                    token_to_index.get(
                        token,
                        UNK_ID,
                    )
                    for token in tokenize(
                        text
                    )
                ],
                dtype=torch.long,
            ),
            "label": torch.tensor(
                0,
                dtype=torch.long,
            ),
            "text": text,
        }
        for text in texts
    ]

    return collate_batch(examples)


feature_texts = [
    "bank approves customer loan",
    "customer requests payment refund",
    "server reports network error",
    "flight arrives at airport",
]

feature_batch = encode_for_local_model(
    feature_texts
)

trained_local_model.eval()

with torch.no_grad():
    feature_output = (
        trained_local_model(
            feature_batch[
                "input_ids"
            ].to(DEVICE),
            feature_batch[
                "padding_mask"
            ].to(DEVICE),
        )
    )

sentence_features = (
    feature_output[
        "pooler_output"
    ].cpu().numpy()
)

print(
    "Sentence feature shape:",
    sentence_features.shape,
)

# 28. Sentence Similarity

In [ ]:
def cosine_similarity(
    left: np.ndarray,
    right: np.ndarray,
) -> float:
    denominator = (
        np.linalg.norm(left)
        * np.linalg.norm(right)
    )

    return float(
        np.dot(left, right)
        / max(
            denominator,
            1e-12,
        )
    )


similarity_matrix = np.zeros(
    (
        len(feature_texts),
        len(feature_texts),
    )
)

for row in range(
    len(feature_texts)
):
    for column in range(
        len(feature_texts)
    ):
        similarity_matrix[
            row,
            column,
        ] = cosine_similarity(
            sentence_features[row],
            sentence_features[column],
        )

pd.DataFrame(
    similarity_matrix,
    index=feature_texts,
    columns=feature_texts,
)

Task-fine-tuned classifier representations are not automatically optimal
semantic-similarity embeddings.

# 29. Saving and Reloading a Checkpoint

Hugging Face models commonly support `save_pretrained()` and
`from_pretrained()`.

The offline model uses standard PyTorch serialization to demonstrate the same
checkpoint lifecycle.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = (
        Path(directory)
        / "local_transformer.pt"
    )

    torch.save(
        {
            "model_state_dict": (
                trained_local_model.state_dict()
            ),
            "vocabulary": vocabulary,
            "classes": list(
                label_encoder.classes_
            ),
        },
        checkpoint_path,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_model = (
        LocalTransformerClassifier(
            vocabulary_size=len(
                checkpoint["vocabulary"]
            ),
            class_count=len(
                checkpoint["classes"]
            ),
        ).to(DEVICE)
    )

    reloaded_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    reload_metrics = evaluate_model(
        reloaded_model,
        test_loader,
    )

print(
    "Reloaded accuracy:",
    round(
        reload_metrics["accuracy"],
        3,
    ),
)

# 30. Optional Pipeline Demonstration

This cell performs no download unless both switches permit it.

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import pipeline

    classifier = pipeline(
        task="text-classification",
        model=DEMO_MODEL_ID,
        device=-1,
        model_kwargs={
            "local_files_only": (
                USE_LOCAL_FILES_ONLY
            )
        },
    )

    pipeline_results = classifier(
        [
            "The experiment worked very well.",
            "The system failed repeatedly.",
        ]
    )

    display(
        pd.DataFrame(
            pipeline_results
        )
    )
else:
    print(
        "Optional pipeline demo skipped. "
        "Install transformers and set "
        "RUN_HUGGING_FACE_DEMOS=True."
    )

# 31. Optional Manual Auto-Class Inference

Manual inference exposes every processing stage.

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        DEMO_MODEL_ID,
        local_files_only=(
            USE_LOCAL_FILES_ONLY
        ),
    )

    hf_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            DEMO_MODEL_ID,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
        .to("cpu")
    )

    encoded = tokenizer(
        [
            "The results are excellent.",
            "The test produced a serious error.",
        ],
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    hf_model.eval()

    with torch.no_grad():
        hf_output = hf_model(
            **encoded
        )

    hf_probabilities = torch.softmax(
        hf_output.logits,
        dim=1,
    )

    print(
        "input_ids:",
        encoded["input_ids"].shape,
    )
    print(
        "attention_mask:",
        encoded[
            "attention_mask"
        ].shape,
    )
    print(
        "logits:",
        hf_output.logits.shape,
    )
    print(
        hf_probabilities
    )
else:
    print(
        "Optional Auto-Class inference skipped."
    )

# 32. Optional Feature-Extraction Demonstration

A base model can return contextual token states without a classification head.

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import (
        AutoModel,
        AutoTokenizer,
    )

    feature_tokenizer = (
        AutoTokenizer.from_pretrained(
            DEMO_MODEL_ID,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
    )

    base_model = (
        AutoModel.from_pretrained(
            DEMO_MODEL_ID,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
        .to("cpu")
    )

    feature_inputs = feature_tokenizer(
        [
            "The bank approved the loan.",
            "The server reported an error.",
        ],
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    base_model.eval()

    with torch.no_grad():
        base_output = base_model(
            **feature_inputs
        )

    contextual_states = (
        base_output.last_hidden_state
    )

    valid = feature_inputs[
        "attention_mask"
    ].unsqueeze(-1)

    mean_features = (
        (
            contextual_states
            * valid
        ).sum(dim=1)
        / valid.sum(
            dim=1
        ).clamp(min=1)
    )

    print(
        "Token states:",
        contextual_states.shape,
    )
    print(
        "Mean-pooled features:",
        mean_features.shape,
    )
else:
    print(
        "Optional feature-extraction demo skipped."
    )

# 33. Fine-Tuning Workflow Structure

A standard sequence-classification fine-tuning workflow is:

1. inspect the model card;
2. load tokenizer;
3. tokenize train, validation, and test data;
4. load `AutoModelForSequenceClassification`;
5. define optimizer and scheduler;
6. train on labeled batches;
7. select a checkpoint with validation metrics;
8. evaluate once on the test set;
9. save tokenizer, model, configuration, and metadata.

In [ ]:
fine_tuning_stages = pd.DataFrame(
    {
        "stage": [
            "Select checkpoint",
            "Tokenize data",
            "Load task head",
            "Train",
            "Validate",
            "Test",
            "Save artifacts",
        ],
        "main_risk": [
            "task or license mismatch",
            "truncation or label misalignment",
            "incorrect number of labels",
            "overfitting",
            "test leakage",
            "repeated test tuning",
            "missing metadata",
        ],
    }
)

fine_tuning_stages

# 34. Trainer Workflow Structure

The Trainer API can manage:

- training loops;
- evaluation;
- checkpointing;
- logging;
- gradient accumulation;
- mixed precision;
- metric callbacks.

The exact argument names can change across library versions, so code should be
checked against the installed Transformers documentation.

In [ ]:
trainer_pseudocode = '''
from transformers import Trainer, TrainingArguments

arguments = TrainingArguments(
    output_dir="checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

trainer.train()
'''

print(trainer_pseudocode)

A manual PyTorch loop remains useful when exact control or version stability is
required.

# 35. Data Collators

Data collators assemble examples into batches.

They may provide:

- dynamic padding;
- label tensors;
- MLM corruption;
- sequence-to-sequence target preparation;
- token-classification label alignment.

In [ ]:
collator_examples = pd.DataFrame(
    [
        ("DefaultDataCollator", "already aligned fields"),
        ("DataCollatorWithPadding", "dynamic tokenizer padding"),
        ("DataCollatorForLanguageModeling", "MLM or causal LM batches"),
        ("DataCollatorForSeq2Seq", "source and target batching"),
        ("DataCollatorForTokenClassification", "token labels"),
    ],
    columns=["Collator", "Typical use"],
)

collator_examples

# 36. Checkpoint Revisions and Caching

Repositories can change. Reproducible experiments should record:

- model ID;
- revision or commit hash;
- tokenizer revision;
- library versions;
- local cache policy;
- trust settings for custom code.

In [ ]:
checkpoint_metadata = pd.Series(
    {
        "model_id": "organization/model-name",
        "revision": "commit-hash-or-tag",
        "local_files_only": True,
        "trust_remote_code": False,
        "device": "cpu",
    }
)

checkpoint_metadata

Enabling custom remote code should be treated as a software-supply-chain
decision, not a routine default.

# 37. CPU, GPU, and Memory Decisions

CPU is appropriate for:

- tokenization demonstrations;
- small-batch inference;
- compact models;
- feature extraction on small datasets;
- notebook instruction.

GPU is useful for:

- repeated large-batch inference;
- fine-tuning medium or large checkpoints;
- long sequences;
- generative models;
- large evaluation datasets.

In [ ]:
hardware_decisions = pd.DataFrame(
    [
        ("Small encoder inference", "CPU"),
        ("Tiny teaching experiment", "CPU"),
        ("Medium-model fine-tuning", "GPU preferred"),
        ("Large decoder generation", "GPU usually required"),
        ("Long-context batch evaluation", "GPU preferred"),
    ],
    columns=["Workload", "Recommended device"],
)

hardware_decisions

For this course, CPU remains the default. A lesson will explicitly state when
GPU testing is required.

# 38. Responsible Model Selection

Before deployment, examine:

- training-data provenance;
- intended and excluded uses;
- language and demographic coverage;
- benchmark relevance;
- bias;
- privacy risk;
- security risk;
- license;
- environmental and computational cost.

In [ ]:
risk_register = pd.DataFrame(
    [
        ("Bias", "compare subgroup performance"),
        ("Privacy", "avoid exposing sensitive text"),
        ("License", "review redistribution and use terms"),
        ("Domain shift", "evaluate target-domain data"),
        ("Hallucination", "ground and verify generation"),
        ("Security", "avoid unreviewed remote code"),
    ],
    columns=["Risk", "Response"],
)

risk_register

# 39. Arabic and Multilingual Considerations

Arabic checkpoint selection should examine:

- Modern Standard Arabic versus dialect coverage;
- fully vocalized versus unvocalized training data;
- clitic and morphology handling;
- tokenizer fragmentation;
- Arabic script normalization;
- code-switching;
- evaluation corpus variety.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized form",
        "Illustrative segmentation",
    ],
)

arabic_examples

A multilingual label does not guarantee equally strong performance across all
languages or varieties.

For fully vocalized Arabic tasks, removing tashkeel changes the input and may
remove lexical or morphological distinctions.

In [ ]:
arabic_model_checks = pd.DataFrame(
    [
        ("Tokenizer", "inspect fragmentation of vocalized forms"),
        ("Corpus", "verify MSA, dialect, and domain coverage"),
        ("Tashkeel", "confirm preservation policy"),
        ("Metrics", "report Arabic-specific test data"),
        ("Errors", "analyze clitics, agreement, and diacritics"),
    ],
    columns=["Check", "Action"],
)

arabic_model_checks

# 40. Reproducibility and Reporting

Report:

- model ID;
- revision;
- model card review date;
- tokenizer;
- preprocessing;
- maximum sequence length;
- padding and truncation;
- batch size;
- device;
- precision;
- frozen or fine-tuned parameters;
- optimizer and scheduler;
- checkpoint-selection metric;
- random seeds;
- software versions;
- test metrics;
- documented limitations.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        "course_model": "offline local Transformer",
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "transformers_available": TRANSFORMERS_AVAILABLE,
        "optional_demos_enabled": RUN_HUGGING_FACE_DEMOS,
        "training_examples": len(X_train),
        "validation_examples": len(X_validation),
        "test_examples": len(X_test),
    },
    name="Lesson 35 execution",
)

reproducibility_metadata

# 41. Knowledge Check

1. What is the Hugging Face Hub?
2. What information should a model card contain?
3. What is a model ID?
4. Why should a checkpoint revision be recorded?
5. What does the pipeline API combine?
6. What are Auto Classes?
7. What do `input_ids` represent?
8. What does `attention_mask` represent?
9. Why are logits not probabilities?
10. How does masked mean pooling work?
11. How do frozen features and fine-tuning differ?
12. What is a data collator?
13. Why can model popularity be misleading?
14. When is GPU execution justified?
15. Which Arabic properties should be checked before selecting a model?

# 42. Exercises

## Exercise 1 — Model Card Review

Select one checkpoint and summarize its intended use, license, languages,
training data, evaluation, and limitations.

## Exercise 2 — Pipeline

Run text classification through a pipeline on CPU.

## Exercise 3 — Manual Inference

Reproduce the pipeline result with AutoTokenizer and AutoModel.

## Exercise 4 — Batching

Compare single-example and batched inference time.

## Exercise 5 — Feature Extraction

Compare CLS and masked-mean pooling.

## Exercise 6 — Frozen Classifier

Train a classifier on frozen pretrained features.

## Exercise 7 — Fine-Tuning

Fine-tune the encoder with a smaller learning rate.

## Exercise 8 — Checkpoint Revisions

Pin a model revision and document it.

## Exercise 9 — Arabic Tokenization

Inspect tokenization of fully vocalized Arabic sentences.

## Exercise 10 — Model Comparison

Compare two checkpoints under identical data and metrics.

## Challenge Exercises

1. Add a Hugging Face `Trainer` fine-tuning experiment.
2. Implement gradual encoder unfreezing.
3. Add mixed precision on a compatible GPU.
4. Publish a complete model card for a fine-tuned checkpoint.
5. Compare multilingual and Arabic-specific encoders.

# 43. Summary and Next Lesson

In this lesson:

- the Hugging Face ecosystem was organized into Hub, Transformers, Hub-client,
  dataset, and evaluation components;
- model repositories, IDs, revisions, and cards were examined;
- model selection was connected to task, language, domain, size, license, and
  risk;
- pipelines and Auto Classes were introduced;
- tokenizer fields, padding, truncation, batching, logits, and probabilities
  were explained;
- contextual feature extraction and pooling were implemented;
- frozen-feature and fine-tuning strategies were compared;
- a complete offline CPU Transformer classification workflow was trained and
  evaluated;
- checkpoint saving and reloading were validated;
- optional real Hugging Face pipeline, Auto-Class, and feature-extraction
  cells were provided;
- Trainer, data collators, caching, and revision control were introduced;
- responsible model selection and Arabic multilingual considerations were
  integrated.

## Next Lesson

**Lesson 36: Fine-Tuning Pretrained Transformers for Text Classification**
develops an end-to-end supervised fine-tuning workflow with tokenized
datasets, label mapping, frozen baselines, full fine-tuning, checkpoint
selection, statistical evaluation, and error analysis.

# References

- Hugging Face Transformers documentation: pipelines, Auto Classes,
  tokenizers, models, and training.
- Hugging Face Hub documentation: model repositories and model cards.
- Vaswani, A. et al. *Attention Is All You Need*.
- Devlin, J. et al. BERT.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.